# CSIS3754 Semester Test 2 – 2024
## Fake Currency Detection – Machine Learning Classifiers

## 2.1 — Import dataset

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

currency = pd.read_csv('fake_currency_data.csv')
print(currency.head())

## 2.2 — Rename Country column

In [ ]:
currency = currency.rename(columns={'Country': 'CountryOfOrigin'})
print(currency.columns.tolist())

## 2.3 — Inspect the data

In [ ]:
# Concise summary
currency.info()

# Statistical summary
print(currency.describe(include='all'))

# Unique values per feature
print(currency.nunique())

## 2.3.1 — Inferences from data inspection

In [ ]:
print("""
Inferences from data inspection:

1. Concise Summary:
   - The dataframe contains a mix of numeric and object (text) columns.
   - Any columns showing non-null counts less than the total row count
     indicate missing values that will need to be addressed.

2. Unique Values:
   - Columns with only 2 unique values (e.g. Counterfeit) are binary
     and can be encoded as [0, 1].
   - Columns with many unique values (e.g. CountryOfOrigin) will require
     one-hot encoding or may introduce high dimensionality.

3. Statistical Summary:
   - The mean, min and max values for numeric features like Weight,
     Length, Width and Thickness can reveal outliers — values that
     are significantly far from the mean suggest outliers are present.
   - A large difference between the mean and median (50%) also indicates
     skewness caused by outliers.

Possible Issues:
   - Outliers are likely present in the numeric features based on the
     statistical summary, which could negatively affect model performance.
   - Missing values may be present and will need to be handled before
     training any classifier.
""")

## 2.4 — Boxplots for numeric features

In [ ]:
numeric_features = ['Weight', 'Length', 'Width', 'Thickness']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, numeric_features):
    sns.boxplot(y=currency[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 2.4.1 — Discuss issue from boxplots

In [ ]:
print("""
The boxplots reveal the presence of outliers in the numeric features
(Weight, Length, Width, Thickness). These are visible as individual
data points plotted beyond the whiskers of the boxplots. Outliers can
negatively affect machine learning model performance by skewing the
decision boundaries, particularly for distance-based models like KNN
and SVM, and should therefore be removed.
""")

## 2.4.2 — Fix outliers using IQR method

In [ ]:
print("""
Method chosen: IQR (Interquartile Range) method.
Any value below Q1 - 1.5*IQR or above Q3 + 1.5*IQR is considered
an outlier and removed. This is the same method used by the boxplot
to identify outliers, making it the most consistent approach.
""")

for col in numeric_features:
    Q1  = currency[col].quantile(0.25)
    Q3  = currency[col].quantile(0.75)
    IQR = Q3 - Q1
    currency = currency[
        (currency[col] >= Q1 - 1.5 * IQR) &
        (currency[col] <= Q3 + 1.5 * IQR)
    ]

print('Shape after removing outliers:', currency.shape)

# Confirm fix by recreating boxplots
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, col in zip(axes, numeric_features):
    sns.boxplot(y=currency[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 2.5 — Reduce sample size

In [ ]:
currency = currency.sample(n=5000, replace=False, random_state=42)
print('Reduced dataframe shape:', currency.shape)

## 2.6 — Pie chart: counterfeit vs genuine

In [ ]:
counts = currency['Counterfeit'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(counts, labels=counts.index,
        autopct=lambda p: f'{p:.1f}%',
        startangle=90)
plt.title('Counterfeit vs Genuine Currency')
plt.show()
print(counts)

## 2.7 — Discuss balance

In [ ]:
print("""
From the pie chart, if the percentage split between counterfeit and
genuine currency is approximately 50/50, the data is balanced and
no further balancing is required. If one class significantly outweighs
the other (e.g. 80/20 split), the data is imbalanced and resampling
techniques such as oversampling the minority class or undersampling
the majority class should be applied to avoid biased model predictions.
""")

## 2.8 — Convert text to numeric

In [ ]:
text_cols = currency.select_dtypes(include=['object']).columns.tolist()

for col in text_cols:
    unique_vals = currency[col].nunique()
    if unique_vals == 2:
        vals = currency[col].unique()
        currency[col] = currency[col].map({vals[0]: 0, vals[1]: 1})
        print(f"Binary encoded: '{col}' -> {vals[0]}=0, {vals[1]}=1")
    elif unique_vals > 2:
        dummies = pd.get_dummies(currency[col], prefix=col)
        currency = pd.concat([currency.drop(columns=[col]), dummies], axis=1)
        print(f"One-hot encoded: '{col}'")

print(currency.head())

## 2.9 — Confirm no object columns remain

In [ ]:
obj_cols = currency.select_dtypes(include=['object']).columns.tolist()
print('Remaining object columns:', obj_cols)
print(currency.dtypes)

## 2.10 — X, y split + train/test split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

currency = currency.astype(
    {col: int for col in currency.select_dtypes('bool').columns}
)
currency = currency.select_dtypes(exclude=['datetime64[ns]'])

X = currency.drop(columns=['Counterfeit'])
y = currency['Counterfeit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)

## 2.11 — Train classifiers + learning curves

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score, learning_curve

models = [
    ('NB',  GaussianNB()),
    ('LR',  LogisticRegression(solver='lbfgs', max_iter=1000)),
    ('SVM', SVC(gamma='auto'))
]

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
results_acc = []
results_f1  = []
names       = []

def plot_learning_curve(model, name, X, y, cv):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, label='Training score', color='blue')
    plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.1, color='blue')
    plt.plot(train_sizes, val_mean, label='CV score', color='orange')
    plt.fill_between(train_sizes, val_mean-val_std, val_mean+val_std, alpha=0.1, color='orange')
    plt.title(f'Learning Curve - {name}')
    plt.xlabel('Training Size')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

for name, model in models:
    acc_scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
    f1_scores  = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='f1_weighted')
    results_acc.append(acc_scores)
    results_f1.append(f1_scores)
    names.append(name)
    print(f'{name} (acc): {acc_scores.mean():.4f}')
    print(f'{name} (f1):  {f1_scores.mean():.4f}')
    print(f'{name} done\n')
    plot_learning_curve(model, name, X_train_scaled, y_train, kfold)

## 2.12 — Best model predictions, confusion matrix & report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

f1_means   = [scores.mean() for scores in results_f1]
best_index = f1_means.index(max(f1_means))
best_name  = names[best_index]
best_clf   = dict(models)[best_name]
print(f'Best model: {best_name} (F1 = {f1_means[best_index]:.4f})')

best_clf.fit(X_train_scaled, y_train)
y_pred = best_clf.predict(X_test_scaled)

print(f'\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_pred))

## 2.13 — Discussion

In [ ]:
print(f"""
=== Model Discussion: {best_name} ===

1. Training Accuracy vs Testing Accuracy:
   - The cross-validated training accuracy reflects how well the model
     performed on the training folds during k-fold validation.
   - If the test accuracy is close to the training accuracy, the model
     generalises well to unseen currency data and is not overfitting.
   - A large gap where training accuracy is much higher than test accuracy
     indicates overfitting — the model memorised training patterns but
     failed to generalise to new currency samples.

2. Precision:
   - Precision is the proportion of predicted counterfeit notes that
     were actually counterfeit. High precision means the model rarely
     flags genuine currency as counterfeit (few false positives).

3. Recall:
   - Recall is the proportion of actual counterfeit notes correctly
     identified. High recall means the model catches most counterfeit
     currency and rarely misses one (few false negatives). In a currency
     detection context, high recall is especially important as missing
     a counterfeit note carries significant risk.

4. F1-Score:
   - The F1-score is the harmonic mean of precision and recall,
     providing a single balanced metric. A high F1-score indicates
     the model performs well at both identifying counterfeit notes
     and avoiding false alarms on genuine currency.
""")